In [1]:
# Load all the 2024 paper samples, create the config files and process and structure the data


import os
import pandas as pd
import squidpy as sq
import scanpy as sc
import multiprocessing
import warnings
import scipy
import json
import numpy as np
from vitessce.data_utils import (
    to_diamond,
    rgb_img_to_ome_zarr,
    optimize_adata,
)



/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/usr/local/lib/python3.10/dist-packages/vitessce/__init__.py:42: UserWarning: Extra installs are necessary to use widgets: No module named 'anywidget'
  warn(f'Extra installs are necessary to use widgets: {e}')
/usr/local/lib/python3.10/dist-packages/vitessce/__init__.py:68: UserWarning: Extra installs are necessary to use exports: No module named 'starlette'
  warn(f'Extra installs are necessary to use exports: {e}')


In [2]:

SPACERANGER_SOURCE_DIR = (
    "/data/zusers/kresgeb/psych_encode/spatialDLPFC/processed-data/rerun_spaceranger"
)
OUTPUT_DIR = "/zata/public_html/users/kresgeb/psych_encode/spatialDLPFC"
TEMPLATE_CONFIG_PATH = "/zata/zippy/kresgeb/psych_screen/paper_data_processing/template_configs/template_config_2024.json"
BAYESSPACE_CLUSTERS_DIR = "/data/zusers/kresgeb/psych_encode/spatialDLPFC/processed-data/rdata/spe/clustering_results"
WHITELIST_PATH = "/zata/zippy/kresgeb/psych_screen/paper_data_processing/whitelist.txt"
FULL_VISIUM_PATH = (
    "/zata/zippy/kresgeb/psych_screen/paper_data_processing/full_visium.h5ad"
)
COLOR_DATA_PATH = (
    "/zata/zippy/kresgeb/psych_screen/paper_data_processing/colors/k16_like_manual.json"
)

# Suppress the specific UserWarnings about unique names
warnings.filterwarnings(
    "ignore",
    message="Variable names are not unique. To make them unique, call `.var_names_make_unique`.",
)
# Suppress the specific UserWarnings about unique names
warnings.filterwarnings(
    "ignore",
    message="Observation names are not unique. To make them unique, call `.obs_names_make_unique`.",
)



In [3]:

# Loosely based on https://github.com/vitessce/vitessce-python/blob/main/demos/human-lymph-node-10x-visium/src/create_zarr.py
def process_sample(sample_name):
    data_output_path = os.path.join(
        OUTPUT_DIR, "data", sample_name, "data.h5ad.zarr")
    image_output_path = os.path.join(
        OUTPUT_DIR, "data", sample_name, "image.ome.zarr")
    source_outs_path = os.path.join(
        SPACERANGER_SOURCE_DIR, sample_name, "outs")

    adata = sq.read.visium(source_outs_path)
    adata.var_names_make_unique()

    # Retrieve the data from the R pipeline used in the paper
    full_visium = sc.read_h5ad(FULL_VISIUM_PATH)
    # The shortened name, ex. Br8667_mid
    sample_id = "_".join(sample_name.split("_")[1:3])
    sample_visium = full_visium[full_visium.obs["sample_id"] == sample_id]

    # Perform normalization
    sc.pp.normalize_total(adata, inplace=True)
    sc.pp.log1p(adata)

    # # Calculate QC metrics
    # adata.var["mt"] = adata.var_names.str.startswith("MT-")
    # sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

    # # Perform basic filtering (much more generous than source)
    # sc.pp.filter_cells(adata, min_genes=100)
    # sc.pp.filter_genes(adata, min_cells=10)
    # adata = adata[adata.obs["pct_counts_mt"] < 30]

    # Remove any remaining spots that the R pipeline in the paper (scran discard) says should be removed
    # NOTE: apparently this still leaves spots that do not have a cluster assigned to them, somehow...
    discard_spots = sample_visium.obs[
        sample_visium.obs["scran_discard"] == "TRUE"
    ].index
    adata = adata[~adata.obs.index.isin(discard_spots)]

    # Clustering
    add_cluster_data(adata, sample_name, k_tuple=(9, 16, 28))

    # Remove all spots that do not have an assigned cluster in BayesSpace k=9
    spots_missing_cluster_data = adata.obs[pd.isna(
        adata.obs["bayes_space_k=9"])].index
    adata = adata[~adata.obs.index.isin(spots_missing_cluster_data)]

    # Determine the top 100 highly variable genes.
    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=100)

    # Genes of Interest are all the highly variable genes and any additional ones in the whitelist provided
    set_genes_of_interest(adata, WHITELIST_PATH)

    # Dimensionality reduction
    sc.pp.pca(adata, mask_var="genes_of_interest")
    sc.pp.neighbors(adata)
    sc.tl.umap(adata)

    # Add manual layers if they exist, store whether it exists or not
    has_manual_layers = add_manual_layers(adata, sample_visium)

    # If there is manual layer data, make sure no Nones/Null make it through (they break the visualization)
    if has_manual_layers:
        spots_missing_manual_layer_data = adata.obs[pd.isna(
            adata.obs["manual_layers"])].index
        adata = adata[~adata.obs.index.isin(spots_missing_manual_layer_data)]

    # Hierarchical clustering of genes for optimal gene ordering
    X_goi_arr = adata[:, adata.var["genes_of_interest"]].X.toarray()
    X_goi_index = adata[:, adata.var["genes_of_interest"]].var.copy().index
    Z = scipy.cluster.hierarchy.linkage(
        X_goi_arr.T, method="average", optimal_ordering=True
    )

    # Get the hierarchy-based ordering of genes.
    num_cells = adata.obs.shape[0]
    goi_index_ordering = scipy.cluster.hierarchy.leaves_list(Z)
    genes_of_interest = X_goi_index.values[goi_index_ordering].tolist()
    all_genes = adata.var.index.values.tolist()
    not_goi = adata.var.loc[~adata.var["genes_of_interest"]
                            ].index.values.tolist()

    def get_orig_index(gene_id):
        return all_genes.index(gene_id)

    var_index_ordering = list(map(get_orig_index, genes_of_interest)) + list(
        map(get_orig_index, not_goi)
    )

    # Create a new *ordered* gene expression dataframe.
    adata = adata[:, var_index_ordering].copy()
    adata.obsm["X_goi"] = adata[:, adata.var["genes_of_interest"]].X.copy()

    # Scale the spatial data to align with the image
    scale_factor = get_scale_factor(sample_name)
    adata.obsm["spatial"] = adata.obsm["spatial"] * scale_factor

    # Create the diamond visualizations for the spots
    adata.obsm["segmentations"] = np.zeros((num_cells, 4, 2))
    radius = 7
    for i in range(num_cells):
        adata.obsm["segmentations"][i, :, :] = to_diamond(
            adata.obsm["spatial"][i, 0], adata.obsm["spatial"][i, 1], radius
        )

    # Write img_arr to OME-Zarr.
    # Need to convert images from interleaved to non-interleaved (color axis should be first).
    img_hires = adata.uns["spatial"][sample_name]["images"]["hires"]
    img_arr = np.transpose(img_hires, (2, 0, 1))
    rgb_img_to_ome_zarr(
        img_arr,
        image_output_path,
        axes="cyx",
        chunks=(1, 256, 256),
        img_name="H & E Image",
    )

    # Optimize and write anndata
    # adata = optimize_adata(
    #     adata,
    #     obs_cols=(["manual_layers"] if has_manual_layers else [])
    #     + ["bayes_space_k=9", "bayes_space_k=16", "bayes_space_k=28"],
    #     var_cols=["highly_variable", "genes_of_interest"],
    #     obsm_keys=["X_goi", "spatial", "segmentations", "X_umap", "X_pca"],
    #     optimize_X=True,
    #     # Vitessce plays nicely with dense matrices saved with chunking
    #     to_dense_X=True,
    # )

    print(adata)
    return adata
    # adata.write_zarr(data_output_path, chunks=[adata.shape[0], 10])

    # Create the config files from the template
    #create_configuration_file(sample_name, has_manual_layers)


def create_configuration_file(sample_name, has_manual_layers=False):
    output_file_path = os.path.join(
        OUTPUT_DIR, "configs", sample_name, "config.json")

    with open(TEMPLATE_CONFIG_PATH, "r") as f:
        data = json.load(f)

    # TODO: This json->String->json thing is gross, and the two places <<Sample_Name>> occurs in the template should be explicitly found and the sample name inserted
    # Adjust for sample name
    # Convert the data to a string
    data_str = json.dumps(data)
    # Replace <<Sample_Name>> with the actual sample name
    data_str = data_str.replace("<<Sample_Name>>", sample_name)
    # Convert the string back to a dictionary
    data = json.loads(data_str)

    if not has_manual_layers:
        # Find the "Manually Annotated Layers" entry in obsSets and remove it
        datasets = data.get("datasets", [])
        for dataset in datasets:
            files = dataset.get("files", [])
            for file in files:
                options = file.get("options", {})
                obs_sets = options.get("obsSets", [])
                options["obsSets"] = [
                    entry
                    for entry in obs_sets
                    if entry["name"] != "Manually Annotated Layers"
                ]

    data = add_color_data(data)

    # Write the updated data to a new JSON file
    with open(output_file_path, "w") as file:
        json.dump(data, file, indent=2)


def hex_to_rgb(hex_color):
    """Converts hex color string to RGB tuple."""
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i: i + 2], 16) for i in (0, 2, 4))


def add_color_data(config_data):
    """
    Fills the 'obsSetColor' section in the config file from the color palette file.

    :param config_path: Path to the config file to be updated.
    :return: Updated config data with filled 'obsSetColor' section.
    """

    # Load the sets color file
    with open(COLOR_DATA_PATH, "r") as sets_file:
        sets_data = json.load(sets_file)

    # Initialize the 'obsSetColor' structure
    obs_set_color = {"A": []}

    # Iterate through each set in the sets color file
    for set_entry in sets_data["sets"]:
        set_name = set_entry["setName"]
        for color_entry in set_entry["colors"]:
            label = color_entry["label"]
            hex_color = color_entry["hex"]
            rgb_color = hex_to_rgb(hex_color)

            # Build the path and color entry for the 'obsSetColor'
            path = [set_name]
            if label:
                path.append(label)

            color_entry = {"path": path, "color": rgb_color}

            # Append to the appropriate place in obsSetColor
            obs_set_color["A"].append(color_entry)

    # Fill the 'obsSetColor' section of the config data
    config_data["coordinationSpace"]["obsSetColor"] = obs_set_color

    return config_data


# Returns True if manual layer data exists
def add_manual_layers(adata, sample_visium):

    manual_layers = sample_visium.obs["manual_layer_label"]

    if manual_layers.notna().any():
        # Replace 'Layer X' with 'LX', but keep 'WM' unchanged
        rename_dict = {f"Layer {i}": f"L{i}" for i in range(1, 7)}
        manual_layers = manual_layers.cat.rename_categories(rename_dict)
        adata.obs["manual_layers"] = adata.obs_names.map(manual_layers)
        return True
    else:
        return False


# TODO improve efficiency, currently opens and searches the clusters.csv once for EACH sample--despite having the data for ALL samples
# This compounds for more entries in k_tuple
# TODO change to save under one obs entry (bayes_space) with multiple columns where each column is a resolution (simpler and MAY be better for performance/compression?)
def add_cluster_data(adata, sample_name, k_tuple=(9,)):
    # The shortened name that exists in the clustering results csv
    # Ex. Br8667_mid
    sample_id = "_".join(sample_name.split("_")[1:3])

    for k in k_tuple:
        # Read the data from clusters.csv
        clusters_path = os.path.join(
            BAYESSPACE_CLUSTERS_DIR, f"bayesSpace_harmony_{k}", "clusters.csv"
        )
        full_cluster_data = pd.read_csv(clusters_path)

        # Extract only the relevant data
        filtered_cluster_data = full_cluster_data[
            full_cluster_data["key"].str.contains(sample_id)
        ].copy()
        filtered_cluster_data.loc[:, "cell_id"] = filtered_cluster_data["key"].apply(
            lambda x: x.split("_")[0]
        )
        filtered_cluster_data = filtered_cluster_data[["cell_id", "cluster"]]
        filtered_cluster_data.set_index("cell_id", inplace=True)

        # Convert to format as found in leiden
        filtered_cluster_data = (
            filtered_cluster_data.astype(int).astype(str).astype("category")
        )

        # Rename the clusters to something more easy for user to understand based on rename_dict
        rename_dict = {
            str(i): f"Spatial Domain {int(k):02d}D{int(i):02d}" for i in range(1, k + 1)
        }
        # uncomment me for renaming
        # filtered_cluster_data["cluster"] = filtered_cluster_data[
        #     "cluster"
        # ].cat.rename_categories(rename_dict)

        # Add the data to the AnnData object
        adata.obs[f"bayes_space_k={k}"] = adata.obs_names.map(
            filtered_cluster_data["cluster"]
        )


def filter_by_scran(adata, sample_visium):
    pass


def set_genes_of_interest(adata, whitelist_path):

    if "highly_variable" in adata.var:
        adata.var["genes_of_interest"] = adata.var["highly_variable"].copy()
    else:
        adata.var["genes_of_interest"] = pd.Series(
            False, index=adata.var.index)
    with open(whitelist_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line.startswith("#") and line in adata.var.index:
                adata.var.loc[line, "genes_of_interest"] = True


def get_scale_factor(sample_name):
    json_path = os.path.join(
        SPACERANGER_SOURCE_DIR, sample_name, "outs", "spatial", "scalefactors_json.json"
    )
    with open(json_path, "r") as f:
        data = json.load(f)
    return data.get("tissue_hires_scalef")


In [4]:
adata = process_sample("DLPFC_Br6522_mid_manual_alignment_all")

/tmp/ipykernel_126777/154104266.py:274: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[f"bayes_space_k={k}"] = adata.obs_names.map(
/usr/local/lib/python3.10/dist-packages/scanpy/preprocessing/_highly_variable_genes.py:696: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["hvg"] = {"flavor": flavor}
2025-05-16 15:15:35.558229: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-16 15:15:35.571359: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747408535.586559  126777 cuda_dnn.cc:8310] Unable t

AnnData object with n_obs × n_vars = 3622 × 36601
    obs: 'in_tissue', 'array_row', 'array_col', 'bayes_space_k=9', 'bayes_space_k=16', 'bayes_space_k=28', 'manual_layers'
    var: 'gene_ids', 'feature_types', 'genome', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'genes_of_interest'
    uns: 'spatial', 'log1p', 'hvg', 'pca', 'neighbors', 'umap'
    obsm: 'spatial', 'X_pca', 'X_umap', 'X_goi', 'segmentations'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'


In [5]:
adata.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 7983267 stored elements and shape (3622, 36601)>

In [6]:
import pandas as pd
import numpy as np

# Set the gene you want to extract
gene_name = "MYH7"  # Change this to "ALB", "MYH7", etc.

# Check if the gene exists
if gene_name not in adata.var_names:
    raise ValueError(f"Gene '{gene_name}' not found in adata.var_names")

# Extract expression for the gene
gene_expr = adata[:, gene_name].X
if hasattr(gene_expr, "toarray"):
    gene_expr = gene_expr.toarray().flatten()
else:
    gene_expr = gene_expr.flatten()

# Check for NaNs
has_nan = np.isnan(gene_expr).any()
if has_nan:
    print(f"⚠️ {gene_name} expression contains NaN values!")
else:
    print(f"✅ No NaN values in {gene_name} expression.")

# Print all non-zero expression values
nonzero_expr = gene_expr[gene_expr != 0]
print(f"\nNon-zero {gene_name} expression values:")
print(nonzero_expr)

# Print summary statistics
print(f"\n{gene_name} expression statistics:")
print(f"Min: {np.nanmin(gene_expr)}")
print(f"Max: {np.nanmax(gene_expr)}")
print(f"Mean: {np.nanmean(gene_expr)}")

# Construct full DataFrame
df = pd.DataFrame({
    "barcode": adata.obs_names,
    gene_name: gene_expr,
    "bayes_space_k=9": adata.obs["bayes_space_k=9"].values,
    "bayes_space_k=16": adata.obs["bayes_space_k=16"].values
})

# Save to CSV
csv_filename = f"{gene_name}_expression_k9_k16.csv"
df.to_csv(csv_filename, index=False)
print(f"\n✅ CSV saved as '{csv_filename}'")

✅ No NaN values in MYH7 expression.

Non-zero MYH7 expression values:
[0.5913785  0.895836   0.67955774 0.39888263 0.49350256 0.9087643
 0.72262836 0.43355283 0.60254425 0.6918113  0.46697545 0.85228884
 0.41544273 0.5853097  0.48865247 0.443053   0.89855826 0.77734107
 0.5092406  0.43570104 0.42791292 0.5025602  0.61010796 1.1630504
 0.7180143  0.8491099  0.52288896 1.1244105  0.4857654  0.51399314
 0.4235249  0.5693023  0.52895015 0.5437913  0.41595474 0.7801346
 0.46073848 0.5272137  0.60731816 0.40101293 0.50859666]

MYH7 expression statistics:
Min: 0.0
Max: 1.1630504131317139
Mean: 0.00689546437934041

✅ CSV saved as 'MYH7_expression_k9_k16.csv'


In [7]:
# Extract SP1 expression values
sp1_expr = adata[:, "SP1"].X
if hasattr(sp1_expr, "toarray"):
    sp1_expr = sp1_expr.toarray().flatten()
else:
    sp1_expr = sp1_expr.flatten()

# Filter and print non-zero values
nonzero_sp1 = sp1_expr[sp1_expr != 0]

print("Non-zero SP1 expression values:")
print(nonzero_sp1)

Non-zero SP1 expression values:
[0.24079758 0.71449673 0.58505243 1.4813037  0.5989242  0.48859316
 0.3620327  0.7231596  0.55686575 0.7672752  0.32545882 0.6485187
 0.68592525 1.5666537  0.73491955 0.49851358 1.165633   0.6013628
 0.62436527 0.402446   0.4819888  0.65678793 0.48147058 0.5526398
 0.32605803 0.8715128  0.5368756  0.5173675  0.40611514 0.80079
 0.5478043  0.5027486  1.1354573  0.8343009  0.5416609  1.1952981
 0.6029087  0.40425196 0.9416238  0.6289069  0.5212605  0.47062504
 1.1960809  0.87527    0.65483534 0.5621832  0.5530976  0.4455928
 0.37687722 0.5887618  0.90087545 0.7680282  0.55647886 0.9290354
 0.71101487 0.7609368  0.53903997 0.6013628  0.6944884  0.35660195
 0.38292003 0.8956274  0.7330003  0.5537092  0.6121707  0.48883036
 0.56865484 0.47663194 0.94023275 0.6054728  0.49937955 1.1240691
 0.25875318 0.6012721  0.57727534 0.47906727 0.42246413 0.78721166
 0.8892095  0.7880063  0.5733837 ]
